# Predicted-GR cascade — SIMPLIFIED gain-prior model (no waveshaper)

**Google Colab**: Runtime → **GPU**. Same pipeline as
`train_lstm_gain_prior_ws_e2e.ipynb` (frozen stage-1 `BlackboxGRLSTM` → this
stage 2); only the stage-2 model is simplified. **Push local changes before running.**

## Why the simplification

Measured on the trained WS e2e run (`e2e_predgr_20260705_185738`,
07_experiments/04 + the 2026-07-07 simplify ablation on the 5 test pairs):

- the **waveshaper learned the identity** — max |W(s)−s| ≈ 3·10⁻⁵, curve
  harmonics ≤ 10⁻⁷, ws_res ≈ 0.003 % of wet RMS; `no_ws == full` to 4 decimals
  on every metric. Dead weight.
- the model's actual work is **Δg** (small GR corrections, mean 0.02–1.6 dB on
  test) plus a **small additive colour** (~3–5 % of wet RMS, > 90 % of its
  energy below 150 Hz).
- **reference point to beat**: on the held-out test pairs the raw predicted-GR
  amplitude match scored *better* than the full WS model
  (ESR-A 0.0540 vs 0.0578, MR-STFT 0.169 vs 0.189, mid-song 55 s segments) —
  stage 2 currently doesn't pay for itself on unseen songs. Fewer moving parts
  makes that diagnosis cleaner.

## The model (`model_gainprior_simple_e2e.GainPriorSimpleE2ELSTM`)

```
s = x · 10^((ĝr + Δg)/20)      Δg = 12·tanh(lin_gain(h))   zero-init
y = s · (1 + m)                m  = tanh(lin_color(h))     zero-init
```

Colour is a **multiply of the gain-matched signal** (signal-locked by
construction: silence in → silence out) instead of a free additive waveform.
Note the honest caveat in the module docstring: (1+m) and 10^(Δg/20) are both
time-varying gains read from the same LSTM state, so the family is
"multiplicative prior + learned time-varying gain"; `USE_MULT_COLOR=False`
collapses to the minimal Δg-only model. 8,322 params (WS run: 8,418).

## Everything else is UNCHANGED from the WS e2e notebook
Same frozen stage-1 run, same v2 seed-42 split (external `test_ground_truth`),
same tvcond knob conditioning, 3 s crops, TBPTT 4410, AdamW + cosine,
100 epochs, bf16, 4c loss. Comparable runs: `e2e_predgr_20260705_185738`
(WS twin), the oracle `gain_prior_ws` run, the from-scratch SOTA LSTM32TVC,
and the untrained zero-init (= predicted-GR amplitude match) baseline.


In [ ]:
# -- 0. Dependencies ---------------------------------------------------
# Identical to the 06_output notebooks (see train_lstm_gain_prior_ws.ipynb
# cell 0 for the rationale behind the pins/stubs).
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} - restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

In [ ]:
# -- 1. Mount Drive (dataset + GR run) + clone repo from GitHub -------
# Code comes from GitHub - push local changes before (re)running this cell.
# Modules resolve from THREE repo dirs: 09_end2end (this pipeline),
# 06_output (dataset/system/model base — must precede any collision), and
# 05_conditioning (loaded BY PATH inside gr_frontend.py, never on sys.path:
# its splits.py/dataset.py/model.py names collide with 06_output's).

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/Diff-SSL-G-Comp"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

E2E_DIR    = os.path.join(REPO_ROOT, "09_end2end")
COND06_DIR = os.path.join(REPO_ROOT, "06_output")
COND05_DIR = os.path.join(REPO_ROOT, "05_conditioning")
for f, d in (("gr_frontend.py", E2E_DIR), ("model_gainprior_ws_e2e.py", E2E_DIR),
             ("model_gainprior_ws.py", COND06_DIR), ("model_blackbox_gr.py", COND05_DIR)):
    assert os.path.isfile(os.path.join(d, f)), f"Clone failed or stale: {d}/{f}. Did you push?"

# frozen GR-predictor run (gr_pred_runs lives next to the dataset on Drive)
GR_RUNS_DIR = os.path.join(os.path.dirname(DRIVE_DATA_ROOT), "gr_pred_runs")
OUTPUT_DIR  = os.path.join(os.path.dirname(DRIVE_DATA_ROOT), "diffssl_e2e_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "gr_curves")), f"Bad DATA_ROOT: {DATA_ROOT}"
assert os.path.isdir(os.path.join(DATA_ROOT, "processed_ground_truth")), "Missing wet audio dir"
assert os.path.isdir(os.path.join(DATA_ROOT, "test_ground_truth")), (
    f"No test_ground_truth/ under {DATA_ROOT} — it defines the held-out test set."
)
assert os.path.isdir(GR_RUNS_DIR), f"No gr_pred_runs/ next to the dataset: {GR_RUNS_DIR}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_tfilm", "model_tfilm", "model_gainprior",
                 "model_gainprior_ws", "model_gainprior_ws_e2e", "gr_frontend",
                 "system_gainprior", "splits", "amplitude_match",
                 "gr_target", "model_blackbox_gr"):
        del sys.modules[_name]

# ORDER MATTERS: 09_end2end first, then 06_output; 05_conditioning stays OFF.
for p in (REPO_ROOT, os.path.join(REPO_ROOT, "nablafx"), COND06_DIR, E2E_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"DATA_ROOT   : {DATA_ROOT}")
print(f"GR_RUNS_DIR : {GR_RUNS_DIR}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

In [ ]:
# -- 2. Cache dataset to Colab local SSD ------------------------------
# Same cache as the 06_output notebooks — dry WAV per song, ORACLE gr_curve
# (.pt) + wet WAV per (song, setting) pair, wet/gr mirrored under their
# original subfolders. The oracle curves are cached too: cell 4 scores the
# predictor against them, and they anchor the quality table in hparams.

import shutil
from dataset_tfilm import discover_gr_pairs

LOCAL_DATA_ROOT = "/content/Diff-SSL-G-Comp"

pairs = discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True)
settings = sorted({p["setting"] for p in pairs})
songs = sorted({p["song"] for p in pairs})
print(f"Caching {len(songs)} songs x {len(settings)} settings ({len(pairs)} pairs) -> {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for song in songs:
    fn = f"{song}_UnmasteredWAV.wav"
    _mirror(Path(DATA_ROOT) / "processed_normalized" / fn,
            Path(LOCAL_DATA_ROOT) / "processed_normalized" / fn)

for p in pairs:
    _mirror(p["gr"], Path(LOCAL_DATA_ROOT) / Path(p["gr"]).relative_to(DATA_ROOT))
    _mirror(p["wet"], Path(LOCAL_DATA_ROOT) / Path(p["wet"]).relative_to(DATA_ROOT))

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

In [ ]:
# -- 3. Imports & hyper-parameters ------------------------------------
# Stage-2 recipe is UNCHANGED from train_lstm_gain_prior_ws.ipynb (ablation
# contract: the GR source is the only intended variable). New knobs are the
# GR-run selection and USE_MATCHED_INPUT.

import importlib
import json
from datetime import datetime

import torch
import lightning as pl
from lightning.pytorch.callbacks import (
    LearningRateMonitor, ModelCheckpoint, TQDMProgressBar,
)
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_tfilm as _dataset_tfilm
importlib.reload(_dataset_tfilm)
from dataset_tfilm import (
    BATCH_SIZE, SAMPLE_LENGTH, SAMPLE_RATE, GRCropDataModule, discover_gr_pairs,
)

import model_tfilm as _model_tfilm
importlib.reload(_model_tfilm)
import model_gainprior_simple_e2e as _model_gainprior_simple_e2e
importlib.reload(_model_gainprior_simple_e2e)
from model_gainprior_simple_e2e import GainPriorSimpleE2ELSTM

import system_gainprior as _system_gainprior   # reused UNCHANGED
importlib.reload(_system_gainprior)
from system_gainprior import GainPriorSystem

import gr_frontend as _gr_frontend
importlib.reload(_gr_frontend)
from gr_frontend import build_view_root, load_frozen_gr_predictor, predict_gr_view

from splits import (
    DIFFSSL_PARAM_RANGES, build_split_manifest, discover_test_ground_truth_keys,
)
from src.dsp import PARAM_ORDER

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -- stage 1: frozen GR predictor (the ONLY new pipeline input) --
GR_RUN_NAME = "lstm_gr_20260705_125142_lstm_blackbox_gr_film"
GR_CKPT     = None      # None -> lowest loss/val among saved best-* (metrics.csv)

# -- split: v2 external test_ground_truth policy --
SPLIT_SEED   = 42
N_VAL_SONGS  = 1

# -- training (unchanged) --
LR               = 1e-3
MAX_EPOCHS       = 100
STEP_NUM_SAMPLES = 4410
SCHEDULER        = "cosine"
ETA_MIN          = 1e-6
USE_AMP          = True
CHECK_VAL_EVERY_N_EPOCH = 1

# -- model core (identical to the oracle WS run) --
HIDDEN_SIZE     = 32
NUM_LAYERS      = 1
NUM_CONTROLS    = 4
TVCOND_DIM      = 16
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
DELTA_MAX_DB    = 12.0
USE_MULT_COLOR  = True    # False -> minimal dg-only model y = x*10^((gr+dg)/20)
USE_MATCHED_INPUT = True  # keep the x*10^(gr/20) LSTM input channel

# -- loss (4c, unchanged) --
TD_WEIGHT      = 0.5
FD_WEIGHT      = 0.5
ENV_WEIGHT     = 0.2
PE_WEIGHT      = 0.1
MRSTFT_VARIANT = "extended"

RUN_TAG    = "diffssl_lstm32_simple_predgr" + ("" if USE_MULT_COLOR else "_dgonly")
RESUME_RUN = None

In [ ]:
# -- 4. Stage 1: predict GR for every pair (frozen, offline) -----------
# Loads the frozen predictor from Drive, asserts its split matches ours
# (test pairs unseen by BOTH stages), then streams every pair's FULL dry
# song (all settings of a song batched in one pass, state carried across
# chunks — cold start only at the true song start) and writes the predicted
# curves as a VIEW dataset root: predicted gr_curves/ + symlinked dry/wet.
# Stage 2 below trains against VIEW_ROOT with zero dataloader changes.

GR_RUN_DIR = os.path.join(GR_RUNS_DIR, GR_RUN_NAME)
assert os.path.isdir(GR_RUN_DIR), f"GR run not on Drive: {GR_RUN_DIR}"

gr_model, gr_hp, gr_ckpt_path = load_frozen_gr_predictor(
    GR_RUN_DIR, COND05_DIR, ckpt_path=GR_CKPT, device=DEVICE)

# -- cross-stage split guard: same seed-42 v2 policy, same external test set --
TEST_KEYS = discover_test_ground_truth_keys(DRIVE_DATA_ROOT)
assert set(gr_hp["test_pair_keys"]) == TEST_KEYS, (
    "GR predictor was trained under a DIFFERENT test split — cascading it "
    f"here would leak. predictor={sorted(gr_hp['test_pair_keys'])} "
    f"vs ours={sorted(TEST_KEYS)}")
assert gr_hp["split_seed"] == SPLIT_SEED and gr_hp["sample_rate"] == SAMPLE_RATE
print(f"Split guard OK: predictor test keys == ours ({len(TEST_KEYS)} pairs)")

VIEW_ROOT = "/content/Diff-SSL-G-Comp-predgr"
build_view_root(VIEW_ROOT, DATA_ROOT)

pairs = discover_gr_pairs(DATA_ROOT, include_test_ground_truth=True)
print(f"Predicting GR for {len(pairs)} pairs -> {VIEW_ROOT}/gr_curves ...")
gr_quality_df = predict_gr_view(
    gr_model, pairs, VIEW_ROOT, DATA_ROOT, device=DEVICE, test_keys=TEST_KEYS)

# predictor quality vs the oracle curves — the cascade inherits this error;
# the post-1s column excludes the song-start cold transient.
summary = gr_quality_df.groupby("split")[["mae_db", "mae_db_post1s"]].mean().round(3)
print("\nPredicted-GR MAE vs oracle curves (dB):")
print(summary)
GR_QUALITY = {f"{r.split}_mae_db_post1s": round(float(r.mae_db_post1s), 4)
              for r in gr_quality_df.groupby("split")[["mae_db_post1s"]]
              .mean().reset_index().itertuples()}
display(gr_quality_df[gr_quality_df.split == "test"])

del gr_model
torch.cuda.empty_cache()

In [ ]:
# -- 5. Preview split (on the predicted-GR view root) ------------------
# Identical policy to the oracle notebook; discovery runs on VIEW_ROOT so the
# manifest's pair inventory is exactly what stage 2 will train on.

preview = build_split_manifest(
    discover_gr_pairs(VIEW_ROOT, include_test_ground_truth=True),
    seed=SPLIT_SEED,
    n_val_songs=N_VAL_SONGS,
    test_pair_keys=TEST_KEYS,
)
leaked = (set(preview.train_pair_keys) | set(preview.val_pair_keys)) & TEST_KEYS
assert not leaked, f"test_ground_truth pairs leaked into train/val: {sorted(leaked)}"

print(f"Settings ({len(preview.all_settings)}): {preview.all_settings}")
print(f"Train songs: {preview.train_songs}")
print(f"Val songs  : {preview.val_songs}")
print(f"Test pairs : {preview.test_pair_keys}")
print(f"Pairs - train={len(preview.train_pair_keys)} "
      f"val={len(preview.val_pair_keys)} test={len(preview.test_pair_keys)}")

In [ ]:
# -- 6. Model size ------------------------------------------------------

model = GainPriorSimpleE2ELSTM(
    num_controls=NUM_CONTROLS, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
    tvcond_dim=TVCOND_DIM, cond_block_size=COND_BLOCK_SIZE, cond_num_layers=COND_NUM_LAYERS,
    delta_max_db=DELTA_MAX_DB, use_mult_color=USE_MULT_COLOR,
    use_matched_input=USE_MATCHED_INPUT,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"GainPriorSimpleE2ELSTM: {n_params:,} params  "
      f"(hidden={HIDDEN_SIZE}, tvcond_dim={TVCOND_DIM}, controls={NUM_CONTROLS}, "
      f"matched_input={USE_MATCHED_INPUT}, delta_max={DELTA_MAX_DB} dB, "
      f"mult_color={USE_MULT_COLOR})")
for name, mod in model.named_children():
    print(f"  {name:10s} {sum(p.numel() for p in mod.parameters()):,}")
print(f"\nWS e2e twin is 8,418; SOTA LSTM32TVC is 8k — parameter-matched. "
      f"Cascade adds the frozen {gr_hp['model']['num_params']:,}-param predictor at inference.")
print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | {SAMPLE_RATE} Hz | "
      f"TBPTT step {STEP_NUM_SAMPLES} | tvcond block {COND_BLOCK_SIZE}")


In [ ]:
# -- 7. DataModule (VIEW root) + zero-init sanity check -----------------
# The datamodule reads the PREDICTED curves. Zero-init identity now means:
# untrained output == amplitude match under the predicted GR — the deployable
# "stage-1-only" baseline. Its crop L1 vs wet is the number training must
# beat; the oracle notebook's same print is the corresponding upper-bound
# start. Expect this one to start worse by roughly the predictor's error.

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert os.path.isdir(os.path.join(VIEW_ROOT, "gr_curves")), "Run cell 4 first."

NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"e2e_predgr_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")
gr_quality_df.to_csv(os.path.join(RUN_DIR, "gr_pred_quality.csv"), index=False)

dm = GRCropDataModule(
    data_root=VIEW_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED, n_val_songs=N_VAL_SONGS,
    test_gt_root=DRIVE_DATA_ROOT,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")

# -- zero-init sanity: untrained model == amplitude match of the GIVEN gr --
from amplitude_match import amplitude_match

if not RESUME_RUN:
    _dry, _gr, _wet, _p = next(iter(dm.val_dataloader()))
    with torch.no_grad():
        model.reset_states()
        _y0 = model(_dry, _gr, _p)
    _diff = float((_y0 - amplitude_match(_dry, _gr)).abs().max())
    _l1 = float(torch.nn.functional.l1_loss(_y0, _wet))
    print(f"zero-init |model - amplitude_match(x, gr_pred)| max = {_diff:.2e}")
    assert _diff < 1e-5, "gain-prior heads are not identity-initialised!"
    print(f"untrained (== predicted-GR amp-match) crop L1 vs wet: {_l1:.6f}  <- training starts here")
    model.reset_states()
    del _dry, _gr, _wet, _p, _y0

In [ ]:
# -- 8. Train ------------------------------------------------------------

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "predgr_cascade: frozen BlackboxGRLSTM -> SIMPLIFIED gain-prior model "
                    "(y = x * 10^((gr_pred + delta_g)/20) * (1 + m); no waveshaper)",
        "model_type": "GainPriorSimpleE2ELSTM",
        "model_ref": "simplified twin of e2e_predgr_20260705_185738 (WS run); waveshaper dropped "
                     "(measured identity), additive color -> multiplicative color"
                     + ("" if USE_MULT_COLOR else "; MULT COLOR OFF (dg-only minimal model)"),
        "gr_frontend": {
            "run": GR_RUN_NAME,
            "ckpt": os.path.basename(gr_ckpt_path),
            "model_type": gr_hp["model_type"],
            "num_params": gr_hp["model"]["num_params"],
            "frozen": True,
            "streaming": "full-song, state carried, cold start at song start only",
            "pred_mae_vs_oracle_db": GR_QUALITY,
        },
        "dataset": "Diff-SSL-G-Comp (predicted-GR view)",
        "setting": "multi (all non-test settings, tvcond on 4 knobs)",
        "conditioning": "knobs via tvcond (TVFiLMCond); PREDICTED GR as multiplicative prior input",
        "use_matched_input": USE_MATCHED_INPUT,
        "sample_rate": SAMPLE_RATE, "sample_length": SAMPLE_LENGTH, "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": PARAM_ORDER, "param_ranges": DIFFSSL_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "external_test_ground_truth (test pairs excluded from train/val by key; "
                        "asserted equal to the GR predictor's split)",
        "test_pair_keys": sorted(TEST_KEYS),
        "train_songs": dm.split.train_songs,
        "val_songs": dm.split.val_songs, "test_songs": dm.split.test_songs,
        "test_settings": dm.split.test_settings,
        "model": {"hidden_size": HIDDEN_SIZE, "num_layers": NUM_LAYERS,
                   "num_controls": NUM_CONTROLS, "tvcond_dim": TVCOND_DIM,
                   "cond_block_size": COND_BLOCK_SIZE, "cond_num_layers": COND_NUM_LAYERS,
                   "delta_max_db": DELTA_MAX_DB, "use_mult_color": USE_MULT_COLOR,
                   "use_matched_input": USE_MATCHED_INPUT,
                   "num_params": n_params},
        "loss": {"td_weight": TD_WEIGHT, "fd_weight": FD_WEIGHT,
                 "env_weight": ENV_WEIGHT, "pe_weight": PE_WEIGHT,
                 "mrstft_variant": MRSTFT_VARIANT,
                 "kind": "td*L1 + fd*MRSTFT + env*envdB_L1 + pe*preemph_L1"},
        "metrics": ["esr", "rmse", "mae", "mse"],
        "optimizer": f"adamw + {SCHEDULER}",
        "scheduler": SCHEDULER, "eta_min": ETA_MIN, "use_amp": USE_AMP,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "training": "diffssl_crop_batches + tbptt_substeps (reset each batch)",
        "lr": LR, "max_epochs": MAX_EPOCHS,
    }, f, indent=2)

system = GainPriorSystem(
    model=model, lr=LR, step_num_samples=STEP_NUM_SAMPLES,
    td_weight=TD_WEIGHT, fd_weight=FD_WEIGHT,
    env_weight=ENV_WEIGHT, pe_weight=PE_WEIGHT, mrstft_variant=MRSTFT_VARIANT,
    scheduler=SCHEDULER, max_epochs=MAX_EPOCHS, eta_min=ETA_MIN, use_amp=USE_AMP,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
                    save_last=True, filename="best-{epoch:03d}-{step}",
                    auto_insert_metric_name=False),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
    check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
)
trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

In [ ]:
# -- 9. Test (external test_ground_truth pairs — unseen by BOTH stages) --
# Directly comparable rows: the oracle gain_prior_ws v2 run (upper bound,
# same test pairs) and the from-scratch SOTA LSTM32TVC. oracle-vs-this gap ==
# the price of predicting GR instead of reading it off the target.

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, datamodule=dm, ckpt_path=best_ckpt)

In [ ]:
# -- 10. Plot: prediction vs target + head shares -----------------------
# Same figure as the oracle notebook, but the bottom panel's "GR input" is
# now the PREDICTED curve — watch whether the learned delta systematically
# bends it (that is Δg doing predictor-error correction, the new work unit
# of this run).

import matplotlib.pyplot as plt
import numpy as np
from system_gainprior import esr_metric

best = torch.load(callbacks[0].best_model_path, map_location="cuda", weights_only=False)
system.load_state_dict(best["state_dict"])
system.eval().cuda()
print(f"Loaded best checkpoint: {callbacks[0].best_model_path}")

val_batches = list(dm.val_dataloader())
dry, gr, wet, params = val_batches[len(val_batches) // 2]

with torch.no_grad():
    system.model.reset_states()
    pred, delta_db, color, ws_res = system.model(
        dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
    pred, delta_db = pred.cpu(), delta_db.cpu()
    color, ws_res = color.cpu(), ws_res.cpu()

dry_np, wet_np, pred_np = dry.numpy(), wet.numpy(), pred.numpy()
gr_np, delta_np = gr.numpy(), delta_db.numpy()
color_np, ws_np = color.numpy(), ws_res.numpy()
n_plots = min(3, dry_np.shape[0])
fig, axes = plt.subplots(2 * n_plots, 1, figsize=(14, 4.6 * n_plots), squeeze=False)
t = np.arange(wet_np.shape[-1]) / SAMPLE_RATE
for r in range(n_plots):
    ax = axes[2 * r, 0]
    ax.plot(t, dry_np[r, 0], label="Dry", alpha=0.35, lw=0.5, color="gray")
    ax.plot(t, wet_np[r, 0], label="Target (wet)", alpha=0.8, lw=0.5)
    ax.plot(t, pred_np[r, 0], label="Predicted", alpha=0.8, lw=0.5)
    pv = torch.from_numpy(pred_np[r]); tv = torch.from_numpy(wet_np[r])
    mae_r = float(np.mean(np.abs(pred_np[r, 0] - wet_np[r, 0])))
    ax.set_title(f"crop {r} - MAE {mae_r:.4f} | ESR {float(esr_metric(tv, pv)):.4f} | "
                 f"mean|dg| {np.abs(delta_np[r]).mean():.3f} dB | "
                 f"mean|color| {np.abs(color_np[r]).mean():.5f}")
    ax.set_ylabel("amp"); ax.legend(loc="lower right", fontsize=8); ax.set_ylim(-1.05, 1.05)

    ax = axes[2 * r + 1, 0]
    ax.plot(t, gr_np[r, 0], label="Predicted GR input (dB)", lw=0.7, color="#1f77b4")
    ax.plot(t, gr_np[r, 0] + delta_np[r, 0], label="GR + learned delta", lw=0.7,
            color="#d62728", alpha=0.8)
    ax.set_ylabel("gain (dB)"); ax.legend(loc="lower right", fontsize=8)
axes[-1, 0].set_xlabel("Time (s)")
fig.suptitle(f"Predicted-GR cascade simple LSTM - best val loss {callbacks[0].best_model_score:.6f}", y=1.002)
fig.tight_layout()
plot_path = os.path.join(RUN_DIR, "eval_output_comparison.png")
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Saved plot -> {plot_path}")
plt.show()

In [ ]:
# -- 11. Colour-multiplier statistics ------------------------------------
# The multiplicative colour head replaces the WS notebook's transfer-curve
# sweep. m = color/gained on a validation batch (exact where gained != 0):
# distribution + relation to compression depth (does colour scale with GR,
# like a real gain stage's level-dependent character?).

if USE_MULT_COLOR:
    with torch.no_grad():
        system.model.reset_states()
        _y, _dg, color_v, _ = system.model(
            dry.cuda(), gr.cuda(), params.cuda(), return_parts_full=True)
        gained_v = _y - color_v
    mask = gained_v.abs() > 1e-4
    m_v = (color_v[mask] / gained_v[mask]).float().cpu().numpy()
    gr_m = gr.cuda()[mask].float().cpu().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.hist(m_v, bins=200, color="#d62728", log=True)
    ax1.set_xlabel("colour multiplier m"); ax1.set_ylabel("count (log)")
    ax1.set_title(f"m distribution  (mean {m_v.mean():+.4f}, p99 |m| {np.quantile(np.abs(m_v), 0.99):.4f})")
    ax1.grid(alpha=0.3)
    ax2.hexbin(gr_m, m_v, gridsize=60, bins="log", cmap="viridis")
    ax2.set_xlabel("predicted GR input (dB)"); ax2.set_ylabel("m")
    ax2.set_title("colour multiplier vs compression depth")
    fig.tight_layout()
    mstats_path = os.path.join(RUN_DIR, "eval_color_multiplier.png")
    fig.savefig(mstats_path, dpi=150, bbox_inches="tight")
    print(f"Saved plot -> {mstats_path}")
    plt.show()
else:
    print("USE_MULT_COLOR=False (dg-only model) — no colour head to analyse.")


In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"